# 02 — XGBoost Surrogate (from scratch)
**Model:** Gradient Boosted Regression Trees — implemented from first principles.

**Why from scratch:**
Building XGBoost manually forces you to understand every piece:
residual fitting, tree structure, learning rate, regularisation.
No `xgboost` or `sklearn` ensemble module is used here.

**XGBoost core idea (simplified):**
```
F₀(x) = mean(y)              # initial prediction
for m = 1..M:
    r = y − Fₘ₋₁(x)         # residuals (pseudo-gradients)
    hₘ = fit a small tree to r
    Fₘ(x) = Fₘ₋₁(x) + η·hₘ(x)   # η = learning rate
```
Each tree corrects what the previous ensemble got wrong.

**Limitation with 17 points:**
Tree methods are not naturally suited to very small 1D datasets.
No uncertainty bands (unlike GP). Use LOO-CV error as the reliability estimate.

**Outputs saved to** `../analysis/`:
- `xgb_predictions.png`
- `xgb_loo_residuals.png`
- `xgb_results.pkl`


---
## Cell 1 — Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle, sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('data_utils.py')))
from data_utils import load_data, plot_loo_residuals, TARGETS, PALETTE
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_absolute_error, r2_score
import warnings; warnings.filterwarnings('ignore')
plt.rcParams.update({'font.size':12,'figure.dpi':120})
print('Imports OK — no xgboost module used')

---
## Cell 2 — Load Data

In [ ]:
DATA_PATH = '../analysis/elastic_constants_fecr.csv'
d = load_data(DATA_PATH)
df          = d['df']
X           = d['X']
X_pred      = d['X_pred']
x_pred_atoms= d['x_pred_atoms']
targets     = d['targets']
noise_gpa   = d['noise_gpa']   # σ per point — three tiers
flagged     = d['flagged']     # bool: Tier C
tier        = d['tier']        # 'A', 'B', 'C'


---
## Cell 3 — XGBoost Implementation from Scratch

### Key components built here:

**DecisionStump** — a depth-1 regression tree (the 'weak learner').
Finds the best single split threshold that minimises MSE on the residuals.

**WeightedDecisionStump** — same but weighted by sample importance.
Used to down-weight the 3 flagged DFT points.

**GradientBoostedTrees** — the ensemble:
- starts from the weighted mean
- each round fits a stump to the current residuals
- applies L2 regularisation on leaf values (λ parameter)
- learning rate η shrinks each tree's contribution

**Hyperparameters (tunable):**
- `n_estimators` — number of boosting rounds
- `eta` — learning rate (smaller = slower but more stable)
- `max_depth` — kept at 1 (stump) because we have only 17 points
- `lam` — L2 regularisation on leaf weights
- `subsample` — fraction of data used per round (set 1.0 here given tiny N)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# DECISION STUMP (depth-1 regression tree)
# ─────────────────────────────────────────────────────────────────────────
class DecisionStump:
    """
    Depth-1 regression tree. Finds best split of 1D input X to minimise
    weighted MSE of residuals r.

    Attributes after fit:
        threshold  : split value
        left_val   : predicted value for X <= threshold
        right_val  : predicted value for X >  threshold
    """
    def __init__(self, lam=1.0):
        self.lam = lam          # L2 regularisation on leaf values
        self.threshold  = None
        self.left_val   = None
        self.right_val  = None

    def _leaf_value(self, g, w):
        """
        Optimal leaf value in XGBoost:
          v* = -sum(g) / (sum(w) + λ)
        where g = residuals, w = sample weights.
        """
        return -np.sum(g) / (np.sum(w) + self.lam) if len(g) > 0 else 0.0

    def _gain(self, g_L, w_L, g_R, w_R):
        """
        XGBoost split gain:
          Gain = ½ [ G_L²/(H_L+λ) + G_R²/(H_R+λ) − (G_L+G_R)²/(H_L+H_R+λ) ]
        where G = sum(g), H = sum(w).
        Gain > 0 means the split reduces the weighted loss.
        """
        def score(g, w): return (np.sum(g)**2) / (np.sum(w) + self.lam) if len(g)>0 else 0.
        return 0.5 * (score(g_L,w_L) + score(g_R,w_R)
                      - score(np.concatenate([g_L,g_R]), np.concatenate([w_L,w_R])))

    def fit(self, X, g, w=None):
        """
        X : (n,1) feature array
        g : (n,)  residuals (negative gradient of L2 loss = y - F(x))
        w : (n,)  sample weights (None = uniform)
        """
        n = len(X)
        if w is None: w = np.ones(n)
        x = X.flatten()
        best_gain, best_thr = -np.inf, None
        # Try every midpoint between adjacent unique values as split
        thresholds = (np.sort(np.unique(x))[:-1] + np.sort(np.unique(x))[1:]) / 2
        for thr in thresholds:
            L = x <= thr;  R = ~L
            if L.sum() == 0 or R.sum() == 0: continue
            gain = self._gain(g[L], w[L], g[R], w[R])
            if gain > best_gain:
                best_gain, best_thr = gain, thr
        if best_thr is None:          # no valid split found
            best_thr = x.mean()
        self.threshold = best_thr
        L = x <= best_thr;  R = ~L
        self.left_val  = self._leaf_value(g[L], w[L])
        self.right_val = self._leaf_value(g[R], w[R])
        return self

    def predict(self, X):
        x = X.flatten()
        return np.where(x <= self.threshold, self.left_val, self.right_val)

print('DecisionStump class defined')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# GRADIENT BOOSTED TREES ENSEMBLE
# ─────────────────────────────────────────────────────────────────────────
class GradientBoostedTrees:
    """
    XGBoost-style gradient boosting with depth-1 trees (stumps).

    Parameters
    ----------
    n_estimators : int   — number of boosting rounds
    eta          : float — learning rate  (0 < η ≤ 1)
    lam          : float — L2 leaf regularisation
    subsample    : float — fraction of data per round (1.0 = use all)
    """
    def __init__(self, n_estimators=100, eta=0.1, lam=1.0, subsample=1.0, seed=42):
        self.n_estimators = n_estimators
        self.eta          = eta
        self.lam          = lam
        self.subsample    = subsample
        self.seed         = seed
        self.trees        = []
        self.F0           = None     # initial prediction
        self.train_loss   = []

    def fit(self, X, y, sample_weight=None):
        rng = np.random.default_rng(self.seed)
        n   = len(y)
        if sample_weight is None:
            sample_weight = np.ones(n)
        sample_weight = sample_weight / sample_weight.sum() * n  # normalise

        # F₀ = weighted mean
        self.F0 = np.average(y, weights=sample_weight)
        F = np.full(n, self.F0)
        self.trees = []

        for m in range(self.n_estimators):
            # Pseudo-residuals (negative gradient of MSE)
            g = y - F                     # g_i = -(dL/dF) = y_i - F(x_i)

            # Subsampling (with replacement)
            if self.subsample < 1.0:
                idx = rng.choice(n, size=int(n*self.subsample), replace=False)
            else:
                idx = np.arange(n)

            stump = DecisionStump(lam=self.lam)
            stump.fit(X[idx], g[idx], sample_weight[idx])
            update = stump.predict(X)
            F = F + self.eta * update
            self.trees.append(stump)
            self.train_loss.append(np.average((y - F)**2, weights=sample_weight))
        return self

    def predict(self, X):
        F = np.full(len(X), self.F0)
        for stump in self.trees:
            F = F + self.eta * stump.predict(X)
        return F

print('GradientBoostedTrees class defined')

---
## Cell 4 — Hyperparameter Search via LOO-CV

With only 17 points, we grid-search over n_estimators and eta,
selecting the combination with lowest LOO-CV MAE per target.

In [ ]:
# Sample weights = inverse noise variance, normalised
# Uses three-tier noise_gpa from CSV, NOT the old binary NOISE_NORMAL/NOISE_FLAGGED
w = 1.0 / (noise_gpa ** 2)          # inverse variance weighting
w = w / w.sum() * len(w)            # normalise so mean weight = 1

print('Sample weights by tier:')
for t in ['A','B','C']:
    m = tier == t
    if m.any():
        σ = noise_gpa[m][0]
        print(f'  Tier {t}: σ={σ:.1f} GPa  →  weight={w[m][0]:.4f}  ({m.sum()} tags)')


---
## Cell 5 — Train Final Models + Full LOO Results

In [ ]:
xgb_models  = {}
xgb_results = {}
for tname in TARGETS:
    y = targets[tname]
    n_est, eta = best_params[tname]
    # Final model on all data
    model = GradientBoostedTrees(n_estimators=n_est, eta=eta, lam=LAM)
    model.fit(X, y, sample_weight=w)
    xgb_models[tname] = model
    # LOO with best params
    mae, r2, loo_preds, loo_true = xgb_loo(X, y, w, n_est, eta, LAM)
    xgb_results[tname] = {
        'mu': model.predict(X_pred),
        'mae': mae, 'r2': r2,
        'loo_true': loo_true, 'loo_preds': loo_preds,
        'residuals': loo_true - loo_preds,
        'best_params': best_params[tname],
        'train_loss': model.train_loss
    }
    print(f'{tname}: n={n_est} η={eta}  LOO MAE={mae:.2f} GPa  R²={r2:.4f}')

---
## Cell 6 — Training Loss Curves

Shows how the ensemble loss decreases with each boosting round.
A flat curve = more rounds not helping. A still-declining curve = could add more rounds.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, tname in zip(axes, TARGETS):
    loss = xgb_results[tname]['train_loss']
    ax.plot(range(1, len(loss)+1), loss, color=PALETTE[tname], lw=2)
    ax.set_xlabel('Boosting round'); ax.set_ylabel('Weighted MSE')
    ax.set_title(f'{tname} — Training Loss')
    ax.grid(alpha=0.3)
plt.suptitle('XGBoost: Training Loss vs Boosting Rounds', fontweight='bold')
plt.tight_layout()
plt.savefig('../analysis/xgb_training_loss.png', bbox_inches='tight')
plt.show(); print('Saved: xgb_training_loss.png')

---
## Cell 7 — Prediction Plots

XGBoost has no native uncertainty output.
The shaded region shows ±MAE (LOO) as a practical error band.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, tname in zip(axes, TARGETS):
    mu  = xgb_results[tname]['mu']
    mae = xgb_results[tname]['mae']
    col = PALETTE[tname]
    ax.fill_between(x_pred_atoms, mu-mae, mu+mae, alpha=0.20, color=col, label=f'±MAE={mae:.1f} GPa')
    ax.plot(x_pred_atoms, mu, '-', color=col, lw=2, label='XGB prediction')
    ax.scatter(df['n_cr'][~flagged], targets[tname][~flagged],
               color='black', s=50, zorder=5, label='DFT')
    ax.scatter(df['n_cr'][flagged],  targets[tname][flagged],
               color='tomato', s=70, marker='D', zorder=5, label='⚠️ flagged')
    ax.set_xlabel('Cr atoms (out of 16)'); ax.set_ylabel(f'{tname} (GPa)')
    ax.set_title(f'{tname} — XGBoost'); ax.legend(fontsize=9)
    ax.grid(alpha=0.3); ax.set_xlim(-0.5, 16.5)
plt.suptitle('XGBoost Surrogate: Fe-Cr Elastic Constants', fontweight='bold')
plt.tight_layout()
plt.savefig('../analysis/xgb_predictions.png', bbox_inches='tight')
plt.show(); print('Saved: xgb_predictions.png')

---
## Cell 8 — LOO-CV Residuals

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, tname in zip(axes, TARGETS):
    res = xgb_results[tname]
    plot_loo_residuals(ax, res['residuals'], df['n_cr'].values, tier,
                       f"{tname} LOO-CV\nMAE={res['mae']:.2f} GPa | R²={res['r2']:.3f}")
plt.suptitle('XGBoost LOO-CV Residuals  (red = flagged)', fontweight='bold')
plt.tight_layout()
plt.savefig('../analysis/xgb_loo_residuals.png', bbox_inches='tight')
plt.show(); print('Saved: xgb_loo_residuals.png')

---
## Cell 9 — Export Results

In [ ]:
with open('../analysis/xgb_results.pkl','wb') as f:
    pickle.dump(xgb_results, f)
print('Saved: ../analysis/xgb_results.pkl')
print('\nXGBoost LOO-CV Summary:')
for t,v in xgb_results.items():
    n,e = v['best_params']
    print(f'  {t}: n_estimators={n}  eta={e}  MAE={v["mae"]:.2f} GPa  R²={v["r2"]:.4f}')